# 回帰Active Learning実践：ターゲット領域を効率よく学ぶ

このNotebookでは、**候補自身の不確実性**と**運用上重要な領域の予測改善**の違いを、`qPosteriorVariance` / `qPosteriorStd` / `qNegIntegratedPosteriorVariance` / EPIGで比較します。

34番の獲得関数概観を踏まえ、35番では**ターゲット領域を定めた実験計画と観測後の変化**に重点を置きます。独自獲得関数は定義しません。

関連: [Active Learning理論](../../docs/theory/acquisition/09_active_learning.md) / [回帰ALガイド](../../docs/optimization/regression_active_learning.md) / [EPIGガイド](../../docs/optimization/epig.md)


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.active_learning import qNegIntegratedPosteriorVariance
from botorch.optim import optimize_acqf

from robotorchan.acquisition import (
    ExpectedPredictiveInformationGain,
    qPosteriorStd,
    qPosteriorVariance,
)
from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 何を最適化するNotebookなのか

Active Learningでは、目的関数の**最大値**を探す代わりに、**学習したい領域の予測を改善する観測点**を選びます。`target_X` は将来の運用・評価に使う入力の集合で、次に実験する候補そのものではありません。

実際には「製品の通常運転範囲の予測精度を改善したい」「実験できる範囲は広いが、精度が必要な範囲は狭い」といった状況を想定します。

## 2. Regression problem

観測点を中央付近に多く置き、未観測領域を残します。ALでは関数値が高い場所ではなく、learning objectiveにとって価値のある場所を選びます。

In [ ]:
def latent_function(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * torch.cos(6 * torch.pi * X)


train_X = torch.tensor([[0.08], [0.28], [0.42], [0.50], [0.58], [0.72], [0.92]])
train_Y = latent_function(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

## 3. Posterior uncertainty

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

## 4. Pointwise uncertainty: qPosteriorVariance and qPosteriorStd

q=1ではvarianceとstandard deviationは単調変換なので、単純なargmaxは同じです。ただしscore scaleが異なるため、他のutilityやpenaltyとの合成では交換可能とは限りません。

In [ ]:
variance_acq = qPosteriorVariance(model)
std_acq = qPosteriorStd(model)

variance_candidate, variance_value = optimize_acqf(
    acq_function=variance_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
std_candidate, std_value = optimize_acqf(
    acq_function=std_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

print("variance candidate:", variance_candidate.item())
print("std candidate:", std_candidate.item())
print("same candidate:", torch.allclose(variance_candidate, std_candidate, atol=1e-4))

### 分散と標準偏差はなぜ似た候補を選ぶのか

標準偏差は分散の平方根なので、**単一点の非負の事後分散を最大化するだけなら**理論上の最大位置は同じです。最適化器の数値誤差や局所解で実際の出力が少し異なる可能性はあります。

これは「2つの獲得関数が同じ単位のスコアを返す」という意味ではありません。分散は応答値の2乗の単位、標準偏差は応答値と同じ単位です。

## 5. Candidate set and target set are different concepts

次は「候補自身が不確実か」ではなく「その候補を観測するとtarget regionをどれだけ学べるか」を考えます。ここではprediction qualityが重要なtarget regionを`x∈[0.35, 0.65]`とします。

In [ ]:
target_X = torch.linspace(0.35, 0.65, 25).unsqueeze(-1)
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.scatter(train_X.squeeze(-1), torch.zeros_like(train_X.squeeze(-1)), label="observed")
ax.scatter(
    target_X.squeeze(-1),
    torch.ones_like(target_X.squeeze(-1)),
    marker="|",
    label="target region",
)
ax.set_yticks([0.0, 1.0], labels=["observed", "target"])
ax.set_xlabel("x")
ax.legend()
plt.show()

### 観測点・ターゲット点・候補点を区別する

- `train_X`：既に測定した点。GPの学習に使います。
- `target_X`：予測を良くしたい点。EPIGやqNIPVの評価対象です。
- `candidate`：次に実験で測定する点。`optimize_acqf` が選びます。

候補点はターゲット領域の外にあっても構いません。GPの相関を通じてターゲット領域の予測が改善される場合があるためです。

## 6. Integrated uncertainty reduction: qNIPV

BoTorch native `qNegIntegratedPosteriorVariance`は、candidate batchを観測した後のtarget points上のintegrated posterior varianceを小さくするように選択します。q=2をjointに最適化するため、高variance点を独立に2個選ぶのとは異なります。

In [ ]:
nipv = qNegIntegratedPosteriorVariance(model=model, mc_points=target_X)
nipv_candidates, nipv_value = optimize_acqf(
    acq_function=nipv,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert nipv_candidates.shape == torch.Size([2, 1])
print("qNIPV joint candidates:", nipv_candidates.squeeze(-1))
print("qNIPV value:", nipv_value)

## 7. Predictive information: EPIG

EPIGはcandidate自身のentropyではなく、candidate observationとtarget predictionsのmutual informationを評価します。現在のrobotorchan実装はq=1・single-output・joint Gaussian posterior・finite target setという明示的contractです。

### qNIPVの2点同時提案

`q=2` は候補2点をまとめて評価することを意味します。事後共分散を利用するため、単純に`qPosteriorVariance`の高い点を2つ並べる方法とは異なります。

`mc_points=target_X` は積分対象を有限点で近似するための入力です。ここでの獲得関数値は「観測後の分散を減らすためのスコア」で、目的関数値ではありません。

In [ ]:
epig = ExpectedPredictiveInformationGain(
    model=model,
    target_X=target_X,
    observation_noise=0.01,
)
epig_candidate, epig_value = optimize_acqf(
    acq_function=epig,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

assert epig_candidate.shape == torch.Size([1, 1])
print("EPIG candidate:", epig_candidate.item())
print("EPIG value:", epig_value.item())

## 8. Compare the decision outputs

3つのcriterionは異なるlearning questionに答えるため、acquisition valueの絶対値を共通scoreとして比較しません。candidate位置をposteriorとtarget region上で確認します。

### EPIGはどんな観測を重視するか

EPIGは、候補の観測結果とターゲット領域での予測との**情報量のつながり**を評価します。観測候補自身の分散だけではなく、運用領域の予測をどれだけ学べるかに注目する点が特徴です。

`observation_noise=0.01` はEPIG計算で仮定する観測ノイズの大きさです。データ生成関数がノイズを含まないことと、獲得関数側でノイズを仮定することは別の設定です。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), std, label="posterior std")
ax.axvspan(0.35, 0.65, alpha=0.12, label="target region")
ax.axvline(variance_candidate.item(), linestyle="--", label="qPosteriorVariance")
for index, x in enumerate(nipv_candidates.squeeze(-1)):
    ax.axvline(x.item(), linestyle=":", label="qNIPV" if index == 0 else None)
ax.axvline(epig_candidate.item(), linestyle="-.", label="EPIG")
ax.set_xlabel("x")
ax.set_ylabel("posterior std")
ax.legend()
plt.show()

## 9. Evaluate one AL candidate and update the dataset

代表としてEPIG candidateを観測します。AL loopでもcandidate生成後は実験・simulationでlabel/responseを取得し、training dataへ追加してsurrogateを更新します。

In [ ]:
candidate = epig_candidate
candidate_Y = latent_function(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
print("candidate:", candidate)
print("observed response:", candidate_Y)
print("updated n:", updated_X.shape[0])

## 10. Verify uncertainty changes in the target region

### 予測の変化を確認する

候補を実際に測定してGPを再学習した後、**ターゲット領域の平均事後分散**を比較します。小さいほどGPがその領域の潜在関数に対して自信を持っていることを表します。

ただし、再学習時のハイパーパラメータ変化もあるため、必ずしも1回の更新で分散が単調減少するとは限りません。

In [ ]:
with torch.no_grad():
    before_var = model.posterior(target_X).variance.mean()
    after_var = updated_model.posterior(target_X).variance.mean()
print("mean target variance before:", before_var.item())
print("mean target variance after:", after_var.item())

### 観測前後のGPを可視化する

次の図では、予測平均と±2標準偏差の帯を観測前後で比較します。背景の中央領域が今回の**予測改善ターゲット**です。

In [ ]:
with torch.no_grad():
    updated_posterior = updated_model.posterior(grid)
    updated_mean = updated_posterior.mean.squeeze(-1)
    updated_std = updated_posterior.variance.clamp_min(0).sqrt().squeeze(-1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, center, uncertainty, title in (
    (axes[0], mean, std, "Before observation"),
    (axes[1], updated_mean, updated_std, "After observation and fit"),
):
    ax.axvspan(0.35, 0.65, alpha=0.1, label="Target region")
    ax.plot(grid.squeeze(-1), latent_function(grid).squeeze(-1), label="True function")
    ax.plot(grid.squeeze(-1), center.detach(), label="Posterior mean")
    ax.fill_between(grid.squeeze(-1), (center - 2 * uncertainty).detach(), (center + 2 * uncertainty).detach(), alpha=0.2)
    ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="Initial observations")
    ax.scatter(candidate.squeeze(-1), candidate_Y.squeeze(-1), marker="x", s=80, label="New observation")
    ax.set(xlabel="x", title=title)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### 次のEPIG候補を提案する

観測後は、**更新済みGP**を使って獲得関数を作り直します。これが「候補提案 → 観測 → 学習 → 次の候補提案」というALループの基本です。ここでは次の提案まで行い、追加の測定はしません。

In [ ]:
next_epig = ExpectedPredictiveInformationGain(
    model=updated_model, target_X=target_X, observation_noise=0.01
)
next_candidate, next_epig_value = optimize_acqf(
    acq_function=next_epig, bounds=bounds, q=1,
    num_restarts=5, raw_samples=64,
)
print("更新後の次のEPIG候補:", next_candidate.item())
print("更新後のEPIGスコア:", next_epig_value.item())
assert updated_X.shape[0] == train_X.shape[0] + 1
assert next_candidate.shape == (1, 1)

## 11. Decision semantics

| Acquisition | Learns what? | Candidate contract |
| --- | --- | --- |
| qPosteriorVariance / Std | candidate自身のmarginal uncertainty | q=1 |
| qNIPV | target region全体のposterior uncertainty | joint q batch |
| EPIG | target predictionsとのinformation dependence | current implementation q=1 |

ALではまず「どこでprediction qualityが必要か」を決めることが重要です。target setがdeployment distributionを表さなければ、target-aware acquisitionも誤った領域を重視します。

## 12. Epistemic and aleatoric uncertainty

追加観測で減らしたいのは主にepistemic uncertaintyです。観測noiseが大きいだけの場所をpredictive varianceだけで追うと、irreducibleなaleatoric uncertaintyを繰り返し観測する可能性があります。

このNotebookはnoise-free synthetic latent functionを使い、criterion間のdecision semanticsに焦点を当てています。heteroskedastic/noise-modelingは専用Notebookを参照してください。

## 13. Scope boundaries

- BOのEI/UCBはoptimum discoveryが目的であり、このNotebookのlearning utilityとは異なります。
- qStraddle / qBoundaryVariance / qRandomizedStraddleはlevel-set・boundary learningとして別Phaseで扱います。
- classificationのPredictiveEntropy、BALD、Margin等はclassification model contractが整うまで対応済みとはしません。
- EPIGをNGBoostやtree ensembleへ適用できるとは主張しません。joint Gaussian covarianceが必要です。
- qNIPVはBoTorch native APIを直接使い、robotorchan wrapperを追加しません。
- stopping criterionはacquisitionとは別責務です。

## 14. Related notebooks

- `01_single_task_gp.ipynb`: single-objective BO loop
- `16_noise_models.ipynb`: observation-noise semantics
- `32_standard_acquisition.ipynb`: BO acquisition comparison

このNotebookはregression Active Learningのdecision workflowを担当します。

## 15. 実務での選択と既存API

| 目的 | 既存獲得関数 | 重要な設定 |
|---|---|---|
| 候補自身の不確実性を調べる | `qPosteriorVariance` / `qPosteriorStd` | `q=1` |
| ターゲット領域の分散を減らす | `qNegIntegratedPosteriorVariance` | `mc_points=target_X`, `q=2` |
| ターゲット予測の情報を増やす | `ExpectedPredictiveInformationGain` | `target_X`, `observation_noise`, `q=1` |
| 観測後にモデルを更新する | `SingleTaskGP.fit()` | 新しい観測データ |
| 次の候補を提案する | `optimize_acqf` | 更新後の獲得関数と`bounds` |

**34番との違い：** 34番は回帰AL獲得関数の全体像（閾値学習のqStraddleも含む）を扱います。35番は、ターゲット領域の予測を改善する実験設計を中心に学びます。